# Preprocessing

In [ ]:
import html
import re
import pandas as pd
from transformers import pipeline
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

ner_pipeline = pipeline(
    "ner",
    model="fahmisyaifudin/indobert_ner_p1",
    tokenizer="fahmisyaifudin/indobert_ner_p1",
    aggregation_strategy="simple",
    device=0,
    stride=64
)
ner_pipeline.tokenizer.model_max_length = 512

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# Standard & EMSCAD Regex Patterns
URL_PAT = r'https?://\S+|www\.\S+'
EMAIL_PAT = r'[\w\.-]+@[\w\.-]+\.\w+'
PHONE_PAT = r'\b(?:\+?62|0)8[1-9](?:[\s-]?\d){7,11}\b'

EMSCAD_EMAIL_HASH = r'#EMAIL_[a-f0-9]+#'
EMSCAD_PHONE_HASH = r'#PHONE_[a-f0-9]+#'
EMSCAD_URL_HASH   = r'#URL_[a-f0-9]+#'

STANDARD_URL_PAT   = r'https?://\S+|www\.\S+'
STANDARD_EMAIL_PAT = r'[\w\.-]+@[\w\.-]+\.\w+'
STANDARD_PHONE_PAT = r'\b(?:\+?\d{1,3}[\s-]?)?\(?\d{2,4}\)?[\s-]?\d{3,4}[\s-]?\d{3,4}\b|\b(?:\+?62|0)8[1-9](?:[\s-]?\d){7,11}\b'

HOAX_NARRATIVE_PATTERNS = [
    r'cek fakta', r'periksa fakta', r'turnbackhoax', r'mafindo',
    r'hoaks', r'hoax', r'disinformasi', r'misinformasi',
    r'diduga', r'postingan(?: foto| video)?', r'unggahan', r'tangkapan layar',
    r'akun (?:facebook|instagram|tiktok|media sosial)', r'beredar(?: sebuah| di media sosial)?',
    r'klaim(?:nya)?', r'narasi(?:nya)?', r'menurut penelusuran', r'faktanya',
    r'#\w+'
]
HOAX_NARRATIVE_RE = re.compile('|'.join(HOAX_NARRATIVE_PATTERNS), re.IGNORECASE)
EMOJI_RE = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]+",
    flags=re.UNICODE
)

def strip_source_artifacts(text: str) -> str:
    """Removes hashtags, emoji, and fact-checking/hoax-narrative boilerplate that leak which site the text was collected from rather than describing the job offer itself."""
    text = EMOJI_RE.sub(' ', text)
    text = HOAX_NARRATIVE_RE.sub(' ', text)
    return text

def mask_and_extract_companies(text: str):
    if not isinstance(text, str) or not text.strip():
        return "", ""

    ner_results = ner_pipeline(text)
    target_tags = {'ORGANIZATION', 'POLITICAL_ORGANIZATION', 'ORG'}
    org_entities = [e for e in ner_results if e.get('entity_group') in target_tags]

    extracted_orgs = []
    for e in org_entities:
        org_str = text[e['start']:e['end']].strip()
        if org_str:
            extracted_orgs.append(org_str)

    unique_orgs = list(dict.fromkeys(extracted_orgs))
    org_entities.sort(key=lambda x: x['start'], reverse=True)

    masked_text = text
    for entity in org_entities:
        start = entity['start']
        end = entity['end']
        masked_text = masked_text[:start] + ' [PERUSAHAAN] ' + masked_text[end:]

    return masked_text, ", ".join(unique_orgs)

def process_scraped_text(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {
            "text_clean_no_contact": "",
            "text_clean_with_contact": "",
            "extracted_emails": "",
            "extracted_phones": "",
            "extracted_urls": "",
            "extracted_companies": ""
        }

    clean = html.unescape(text)
    clean = re.sub(r'<[^>]+>', ' ', clean)
    clean = strip_source_artifacts(clean)

    urls = list(dict.fromkeys(re.findall(URL_PAT, clean)))
    emails = list(dict.fromkeys(re.findall(EMAIL_PAT, clean)))
    phones = list(dict.fromkeys([re.sub(r'[\s-]', '', p) for p in re.findall(PHONE_PAT, clean)]))

    masked = re.sub(URL_PAT, ' [URL] ', clean)
    masked = re.sub(EMAIL_PAT, ' [EMAIL] ', masked)
    masked = re.sub(PHONE_PAT, ' [NOMOR_HP] ', masked)
    masked, companies = mask_and_extract_companies(masked)

    no_contact = re.sub(r'\s+', ' ', masked).strip()
    with_contact = re.sub(r'\s+', ' ', clean).strip()

    return {
        "text_clean_no_contact": no_contact,
        "text_clean_with_contact": with_contact,
        "extracted_emails": ", ".join(emails),
        "extracted_phones": ", ".join(phones),
        "extracted_urls": ", ".join(urls),
        "extracted_companies": companies
    }

def process_emscad_text(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {
            "text_clean_no_contact": "",
            "text_clean_with_contact": "",
            "extracted_emails": "",
            "extracted_phones": "",
            "extracted_urls": "",
            "extracted_companies": ""
        }

    clean = html.unescape(text)
    clean = re.sub(r'<[^>]+>', ' ', clean)
    clean = strip_source_artifacts(clean)

    emscad_emails = re.findall(EMSCAD_EMAIL_HASH, clean, re.IGNORECASE)
    std_emails = re.findall(STANDARD_EMAIL_PAT, clean)
    emails = list(dict.fromkeys(emscad_emails + std_emails))

    emscad_phones = re.findall(EMSCAD_PHONE_HASH, clean, re.IGNORECASE)
    std_phones = re.findall(STANDARD_PHONE_PAT, clean)
    phones = list(dict.fromkeys(emscad_phones + std_phones))

    emscad_urls = re.findall(EMSCAD_URL_HASH, clean, re.IGNORECASE)
    std_urls = re.findall(STANDARD_URL_PAT, clean)
    urls = list(dict.fromkeys(emscad_urls + std_urls))

    masked = clean
    masked = re.sub(EMSCAD_URL_HASH, ' [URL] ', masked, flags=re.IGNORECASE)
    masked = re.sub(STANDARD_URL_PAT, ' [URL] ', masked)
    masked = re.sub(EMSCAD_EMAIL_HASH, ' [EMAIL] ', masked, flags=re.IGNORECASE)
    masked = re.sub(STANDARD_EMAIL_PAT, ' [EMAIL] ', masked)
    masked = re.sub(EMSCAD_PHONE_HASH, ' [NOMOR_HP] ', masked, flags=re.IGNORECASE)
    masked = re.sub(STANDARD_PHONE_PAT, ' [NOMOR_HP] ', masked)
    masked, companies = mask_and_extract_companies(masked)

    no_contact = re.sub(r'\s+', ' ', masked).strip()
    with_contact = re.sub(r'\s+', ' ', clean).strip()

    return {
        "text_clean_no_contact": no_contact,
        "text_clean_with_contact": with_contact,
        "extracted_emails": ", ".join(emails),
        "extracted_phones": ", ".join(phones),
        "extracted_urls": ", ".join(urls),
        "extracted_companies": companies
    }

# Process datasets directly in memory with tqdm progress bars
df1_raw = pd.read_excel('MERGED_Job_Offer_Clean.xlsx')
res1_data = [process_scraped_text(text) for text in tqdm(df1_raw['text_utama'], desc="Processing Scraped Jobs")]
res1 = pd.DataFrame(res1_data)
df1 = pd.concat([df1_raw, res1], axis=1)

df2_raw = pd.read_csv('Dataset_Preprocessed_ID - Dataset_Preprocessed_ID.csv')
res2_data = [process_emscad_text(text) for text in tqdm(df2_raw['text_utama_id'], desc="Processing EMSCAD Jobs")]
res2 = pd.DataFrame(res2_data)
df2 = pd.concat([df2_raw, res2], axis=1)

if 'label' not in df2.columns and 'fraudulent_raw' in df2.columns:
    df2['label'] = df2['fraudulent_raw'].map({1: 'fraud', 0: 'valid', '1': 'fraud', '0': 'valid'})


Processing Scraped Jobs:   0%|          | 0/2717 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing EMSCAD Jobs:   0%|          | 0/2054 [00:00<?, ?it/s]

In [ ]:
import os

if os.path.exists('COMBINED_Job_Offer_Dataset.xlsx'):
    os.remove('COMBINED_Job_Offer_Dataset.xlsx')

df1['source'] = 'scrapped'
df2['source'] = 'emscad'

df1 = df1.loc[:, ~df1.columns.duplicated()].copy()
df2 = df2.loc[:, ~df2.columns.duplicated()].copy()

if 'text_utama' not in df2.columns or df2['text_utama'].isna().all():
    df2['text_utama'] = df2['text_utama_id']

# EMSCAD has no JobStreet/turnbackhoax split, so it gets a single constant value
df2['sumber_dataset'] = 'emscad'

target_columns = [
    'text_utama', 'text_clean_with_contact', 'text_clean_no_contact',
    'extracted_emails', 'extracted_phones', 'extracted_urls',
    'extracted_companies', 'label', 'source', 'sumber_dataset'
]

df1_aligned = df1[target_columns].copy()
df2_aligned = df2[target_columns].copy()

df1_modelling, df1_system_eval = train_test_split(
    df1_aligned, test_size=200, stratify=df1_aligned['label'], random_state=42
)

df_modelling = pd.concat([df1_modelling, df2_aligned], ignore_index=True)
df_modelling = df_modelling.sample(frac=1, random_state=42).reset_index(drop=True)

fraud_rate_by_source = df_modelling.groupby('source')['label'].apply(lambda s: (s == 'fraud').mean())
print("Fraud rate per source:")
print(fraud_rate_by_source)

composition = df_modelling.groupby(['label', 'source']).size().unstack(fill_value=0)
print("\nComposition per label x source:")
print(composition)

output_file = 'COMBINED_Job_Offer_Dataset.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_modelling.to_excel(writer, sheet_name='modelling', index=False)
    df1_system_eval.to_excel(writer, sheet_name='system eval', index=False)

print(f"\nModelling set rows: {len(df_modelling)}")
print(f"System evaluation set rows: {len(df1_system_eval)}")
print(f"Label distribution in modelling tab:\n{df_modelling['label'].value_counts()}")

Fraud rate per source:
source
emscad      0.421616
scrapped    0.182757
Name: label, dtype: float64

Composition per label x source:
source  emscad  scrapped
label                   
fraud      866       460
valid     1188      2057

Modelling set rows: 4571
System evaluation set rows: 200
Label distribution in modelling tab:
label
valid    3245
fraud    1326
Name: count, dtype: int64


# Modelling


In [ ]:
! pip install -q transformers datasets evaluate accelerate scikit-learn imbalanced-learn huggingface_hub torch pandas numpy tqdm

In [ ]:
import os
import re
import json
import html
import random
import pathlib
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack, csr_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset
from huggingface_hub import HfApi

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

## Rekayasa Fitur Tambahan dan Ekstraksi Entitas


In [ ]:
def extract_entities_and_features(text_no_contact: str, emails: list = None, phones: list = None, urls: list = None, companies: list = None) -> dict:
    cleaned = text_no_contact if isinstance(text_no_contact, str) else ""

    company_heuristic = r'\b(?:PT|CV|Perum|UD|Firma|Yayasan)\b\.?\s+[A-Z][a-zA-Z0-9\.\s]+'

    emails = [e.strip() for e in (emails or []) if e and e.strip()]
    phones = [p.strip() for p in (phones or []) if p and p.strip()]
    urls = [u.strip() for u in (urls or []) if u and u.strip()]

    # Defensive extra pass in case a company mention slipped past NER masking
    heuristic_companies = [c.strip() for c in re.findall(company_heuristic, cleaned)]
    ner_companies = [c.strip() for c in (companies or []) if c and c.strip()]
    companies_all = list(dict.fromkeys(heuristic_companies + ner_companies))

    suspicious_keywords = ["transfer", "administrasi", "gaji tinggi tanpa syarat", "biaya pendaftaran", "hubungi wa"]
    lower_text = cleaned.lower()
    keyword_counts = {kw: lower_text.count(kw) for kw in suspicious_keywords}
    suspicious_keyword_total = sum(keyword_counts.values())

    digit_count = sum(c.isdigit() for c in cleaned)
    special_char_ratio = sum(not c.isalnum() and not c.isspace() for c in cleaned) / (len(cleaned) + 1e-5)
    text_length = len(cleaned)

    reasons = []
    if suspicious_keyword_total > 0:
        found_kws = [k for k, v in keyword_counts.items() if v > 0]
        reasons.append(f"Terdapat kata kunci mencurigakan: {', '.join(found_kws)}")
    if digit_count > 15:
        reasons.append("Frekuensi karakter angka tinggi (potensi nomor rekening/kontak tak resmi)")
    if special_char_ratio > 0.08:
        reasons.append("Rasio karakter khusus melebihi batas wajar")
    if companies_all:
        reasons.append(f"Entitas perusahaan terdeteksi: {', '.join(companies_all)}")
    if phones:
        reasons.append(f"Nomor telepon terdeteksi: {', '.join(phones)}")
    if emails:
        reasons.append(f"Alamat email terdeteksi: {', '.join(emails)}")
    if urls:
        reasons.append(f"URL/tautan terdeteksi: {', '.join(urls)}")

    return {
        "extracted_phones": ", ".join(phones),
        "extracted_emails": ", ".join(emails),
        "extracted_urls": ", ".join(urls),
        "extracted_companies": ", ".join(companies_all),
        "text_length": text_length,
        "digit_count": digit_count,
        "special_char_ratio": special_char_ratio,
        "suspicious_keyword_count": suspicious_keyword_total,
        "num_emails": len(emails),
        "num_phones": len(phones),
        "num_urls": len(urls),
        "has_contact_info": int(bool(emails or phones or urls)),
        "reasons": reasons
    }


## Baseline Klasik sebagai Ablation Study

TF-IDF dengan Logistic Regression dan SVM linear dijalankan sebagai baseline sebelum fine-tuning IndoBERT, untuk mengukur seberapa jauh model klasik sudah bisa memisahkan kelas hanya dari representasi kata. `run_baseline_ablation` menerima `vocabulary` opsional, sehingga baseline yang sama dapat dijalankan baik dengan kosakata TF-IDF penuh maupun dengan kosakata yang sudah disaring (lihat bagian mitigasi confound di bawah).


In [ ]:
def run_baseline_ablation(X_train_text, y_train, X_test_text, y_test,
                           X_train_extra=None, X_test_extra=None, vocabulary=None, label=""):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), vocabulary=vocabulary)
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_test_vec = vectorizer.transform(X_test_text)

    if X_train_extra is not None and X_test_extra is not None:
        scaler = StandardScaler(with_mean=False)
        X_train_extra_scaled = scaler.fit_transform(X_train_extra)
        X_test_extra_scaled = scaler.transform(X_test_extra)
        X_train_combined = hstack([X_train_vec, csr_matrix(X_train_extra_scaled)])
        X_test_combined = hstack([X_test_vec, csr_matrix(X_test_extra_scaled)])
    else:
        X_train_combined = X_train_vec
        X_test_combined = X_test_vec

    smote = SMOTE(random_state=SEED)
    X_train_res, y_train_res = smote.fit_resample(X_train_combined, y_train)

    baselines = {
        "Logistic Regression + SMOTE": LogisticRegression(random_state=SEED, max_iter=5000),
        "SVM (Linear) + SMOTE": SVC(kernel='linear', random_state=SEED)
    }

    baseline_results = {}
    for name, model in baselines.items():
        model.fit(X_train_res, y_train_res)
        preds = model.predict(X_test_combined)

        report = classification_report(y_test, preds, labels=[0, 1], target_names=["valid", "fraud"], output_dict=True, zero_division=0)
        print(f"\nClassification Report - {name}{label}")
        print(classification_report(y_test, preds, labels=[0, 1], target_names=["valid", "fraud"], zero_division=0))

        baseline_results[name] = {
            "accuracy": float(report["accuracy"]),
            "precision_fraud": float(report["fraud"]["precision"]),
            "recall_fraud": float(report["fraud"]["recall"]),
            "macro_f1": float(report["macro avg"]["f1-score"]),
            "classification_report": report
        }

    return baseline_results


## Pembagian Data dan Menjalankan Baseline

Label kelas (`valid`/`fraud`) dipetakan ke bentuk numerik (0/1) segera setelah data dimuat, agar konsisten digunakan baik oleh baseline klasik maupun IndoBERT. Data dipisahkan berdasarkan sumber sebelum pembagian train/val/test: hold-out test set sebesar 20% hanya diambil dari data `scrapped`, sehingga evaluasi akhir merepresentasikan kondisi produksi (teks asli hasil scraping, bukan EMSCAD). Data `emscad` hanya digunakan untuk memperkaya data latih (`df_train_val`) dan tidak pernah masuk ke set validasi maupun test.


In [ ]:
df = pd.read_excel("COMBINED_Job_Offer_Dataset.xlsx", sheet_name='modelling')

label_map = {"valid": 0, "fraud": 1}
df["label"] = df["label"].map(label_map)

df_scrapped = df[df["source"] == "scrapped"].reset_index(drop=True)
df_emscad = df[df["source"] == "emscad"].reset_index(drop=True)

df_train_val_scrapped, df_test = train_test_split(
    df_scrapped, test_size=0.20, random_state=SEED, stratify=df_scrapped["label"]
)
df_train_val_scrapped = df_train_val_scrapped.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df_train_val = pd.concat([df_train_val_scrapped, df_emscad], ignore_index=True)

baseline_metrics = run_baseline_ablation(
    df_train_val["text_clean_no_contact"], df_train_val["label"],
    df_test["text_clean_no_contact"], df_test["label"],
    label=" (Full Vocabulary)"
)
print("Baseline Ablation Results (full vocabulary):", json.dumps({k: {m: v for m, v in r.items() if m != "classification_report"} for k, r in baseline_metrics.items()}, indent=2))



Classification Report - Logistic Regression + SMOTE (Full Vocabulary)
              precision    recall  f1-score   support

       valid       1.00      0.99      1.00       412
       fraud       0.96      1.00      0.98        92

    accuracy                           0.99       504
   macro avg       0.98      1.00      0.99       504
weighted avg       0.99      0.99      0.99       504


Classification Report - SVM (Linear) + SMOTE (Full Vocabulary)
              precision    recall  f1-score   support

       valid       1.00      0.99      0.99       412
       fraud       0.97      0.98      0.97        92

    accuracy                           0.99       504
   macro avg       0.98      0.99      0.98       504
weighted avg       0.99      0.99      0.99       504

Baseline Ablation Results (full vocabulary): {
  "Logistic Regression + SMOTE": {
    "accuracy": 0.9920634920634921,
    "precision_fraud": 0.9583333333333334,
    "recall_fraud": 1.0,
    "macro_f1": 0.9869226

## Diagnostik Confound Sumber Data (`sumber_dataset`)

Selain sumber `scrapped` vs `emscad`, data `scrapped` sendiri berasal dari dua sumber pengumpulan dengan gaya penulisan yang sangat berbeda: listing resmi JobStreet (`valid`) dan artikel debunking turnbackhoax.id (`fraud`). Karena setiap baris `fraud` selalu berasal dari turnbackhoax.id dan setiap baris `valid` selalu dari JobStreet, kolom sumber pengumpulan ini identik dengan label itu sendiri. Model berisiko belajar membedakan format sumber (panjang teks, gaya penulisan artikel, dsb.) alih-alih pola penipuan yang sesungguhnya. Sel berikut mengukur seberapa parah confound ini pada data latih.


In [ ]:
sumber_label_crosstab = df_train_val_scrapped.groupby("sumber_dataset")["label"].value_counts().unstack(fill_value=0)
print("Distribusi label per sumber pengumpulan (dalam data scrapped, hanya train/val):")
print(sumber_label_crosstab)

text_length_by_sumber = df_train_val_scrapped.assign(text_length=df_train_val_scrapped["text_clean_no_contact"].str.len())
print("\nRata-rata panjang teks per sumber pengumpulan:")
print(text_length_by_sumber.groupby("sumber_dataset")["text_length"].mean())


Distribusi label per sumber pengumpulan (dalam data scrapped, hanya train/val):
label                                0    1
sumber_dataset                             
FAKE_Job_Offer (turnbackhoax.id)     0  368
REAL_Job_Offer (JobStreet)        1645    0

Rata-rata panjang teks per sumber pengumpulan:
sumber_dataset
FAKE_Job_Offer (turnbackhoax.id)     371.728261
REAL_Job_Offer (JobStreet)          1833.143465
Name: text_length, dtype: float64


## Mitigasi: Seleksi Fitur Domain-Invariant

Karena `sumber_dataset` dan `label` berkorelasi sempurna di dalam data `scrapped`, tidak ada cara statistik untuk memisahkan "kata yang menandakan penipuan" dari "kata yang menandakan gaya penulisan turnbackhoax" hanya dari data itu sendiri. Data `emscad` dipakai sebagai domain pembanding karena labelnya tidak terikat pada satu gaya penulisan tertentu. Sebuah n-gram hanya dipertahankan sebagai fitur jika arah asosiasinya terhadap fraud (dibanding valid) konsisten di kedua domain dan cukup sering muncul di keduanya; n-gram yang hanya kuat di satu domain (indikasi artefak sumber, bukan sinyal penipuan asli) dibuang. Baseline dijalankan ulang dengan kosakata yang sudah disaring ini dan dibandingkan dengan hasil kosakata penuh di atas — selisih besar di antara keduanya menandakan seberapa besar hasil sebelumnya mengandalkan jalan pintas sumber data.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_domain_invariant_vocab(df_a, df_b, text_col="text_clean_no_contact", label_col="label", ngram_range=(1, 2), max_features=20000, min_df_per_domain=5):
    count_vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=max_features)
    count_vectorizer.fit(pd.concat([df_a[text_col], df_b[text_col]], ignore_index=True))
    vocab = count_vectorizer.get_feature_names_out()

    def fraud_log_odds_and_doc_freq(df_subset):
        X = count_vectorizer.transform(df_subset[text_col])
        y = df_subset[label_col].values
        fraud_counts = np.asarray(X[y == 1].sum(axis=0)).ravel() + 1
        valid_counts = np.asarray(X[y == 0].sum(axis=0)).ravel() + 1
        log_odds = np.log(fraud_counts / fraud_counts.sum()) - np.log(valid_counts / valid_counts.sum())
        doc_freq = np.asarray((X > 0).sum(axis=0)).ravel()
        return log_odds, doc_freq

    log_odds_a, doc_freq_a = fraud_log_odds_and_doc_freq(df_a)
    log_odds_b, doc_freq_b = fraud_log_odds_and_doc_freq(df_b)

    same_direction = np.sign(log_odds_a) == np.sign(log_odds_b)
    present_in_both = (doc_freq_a >= min_df_per_domain) & (doc_freq_b >= min_df_per_domain)
    keep_mask = same_direction & present_in_both

    return [vocab[i] for i in range(len(vocab)) if keep_mask[i]]

invariant_vocab = get_domain_invariant_vocab(df_train_val_scrapped, df_emscad)
print(f"Domain-invariant vocabulary: {len(invariant_vocab)} n-gram terms retained (out of the full TF-IDF space)")

baseline_metrics_invariant = run_baseline_ablation(
    df_train_val["text_clean_no_contact"], df_train_val["label"],
    df_test["text_clean_no_contact"], df_test["label"],
    vocabulary=invariant_vocab, label=" (Domain-Invariant Vocabulary)"
)
print("Baseline Ablation Results (domain-invariant vocabulary):", json.dumps({k: {m: v for m, v in r.items() if m != "classification_report"} for k, r in baseline_metrics_invariant.items()}, indent=2))


Domain-invariant vocabulary: 2224 n-gram terms retained (out of the full TF-IDF space)

Classification Report - Logistic Regression + SMOTE (Domain-Invariant Vocabulary)
              precision    recall  f1-score   support

       valid       0.98      0.95      0.97       412
       fraud       0.81      0.92      0.86        92

    accuracy                           0.95       504
   macro avg       0.90      0.94      0.91       504
weighted avg       0.95      0.95      0.95       504


Classification Report - SVM (Linear) + SMOTE (Domain-Invariant Vocabulary)
              precision    recall  f1-score   support

       valid       0.98      0.94      0.96       412
       fraud       0.78      0.91      0.84        92

    accuracy                           0.94       504
   macro avg       0.88      0.93      0.90       504
weighted avg       0.94      0.94      0.94       504

Baseline Ablation Results (domain-invariant vocabulary): {
  "Logistic Regression + SMOTE": {
    "a

## Diagnostik Kebocoran Sumber Data (Source Leakage Check)

Dataset modelling menggabungkan dua sumber dengan karakteristik berbeda: data hasil scraping (`source == "scrapped"`) dan EMSCAD (`source == "emscad"`, teks berbahasa Inggris asli yang sudah dianonimkan/dihash sejak sumbernya, sehingga hampir tidak pernah memiliki nomor telepon atau email asli). Jika distribusi label berbeda jauh antar sumber, model dapat mencapai akurasi tinggi hanya dengan mempelajari "dari sumber mana teks ini berasal", bukan pola penipuan yang sesungguhnya. Tiga pengecekan berikut dijalankan sebelum mempercayai hasil baseline:

1. Distribusi label per sumber data.
2. Seberapa mudah sumber data (scrapped vs emscad) dapat ditebak hanya dari teks — jika sangat mudah, ruang fitur TF-IDF kemungkinan mengandung sinyal identitas sumber, bukan sinyal penipuan.
3. Evaluasi lintas sumber: model dilatih pada satu sumber lalu diuji pada sumber lain, untuk melihat apakah performa bertahan di luar sumber pelatihannya.


In [ ]:
print("Distribusi label per sumber data:")
label_by_source = df.groupby("source")["label"].value_counts(normalize=True).unstack()
print(label_by_source)

source_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_source_vec = source_vectorizer.fit_transform(df["text_clean_no_contact"])

source_train_idx, source_test_idx = train_test_split(
    df.index, test_size=0.20, random_state=SEED, stratify=df["source"]
)

source_model = LogisticRegression(random_state=SEED, max_iter=5000)
source_model.fit(X_source_vec[source_train_idx], df.loc[source_train_idx, "source"])
source_preds = source_model.predict(X_source_vec[source_test_idx])
source_leak_accuracy = accuracy_score(df.loc[source_test_idx, "source"], source_preds)

print("\nSeberapa mudah sumber data ditebak dari teks (source-prediction check):")
print(classification_report(df.loc[source_test_idx, "source"], source_preds, zero_division=0))
print(f"Source-prediction accuracy: {source_leak_accuracy:.2%}")

fraud_rate_by_source = df.groupby("source")["label"].apply(lambda s: (s == 1).mean())
print("\nFraud rate per source:")
print(fraud_rate_by_source)

composition = df.groupby(["label", "source"]).size().unstack(fill_value=0)
print("\nComposition per label x source:")
print(composition)

def evaluate_cross_source(train_source, test_source):
    train_df = df[df["source"] == train_source]
    test_df = df[df["source"] == test_source]

    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train = vec.fit_transform(train_df["text_clean_no_contact"])
    X_test = vec.transform(test_df["text_clean_no_contact"])

    clf = LogisticRegression(random_state=SEED, max_iter=5000, class_weight="balanced")
    clf.fit(X_train, train_df["label"])
    preds = clf.predict(X_test)

    print(f"\nEvaluasi Lintas Sumber: latih pada '{train_source}', uji pada '{test_source}'")
    print(classification_report(test_df["label"], preds, labels=[0, 1], target_names=["valid", "fraud"], zero_division=0))

for train_source, test_source in [("scrapped", "emscad"), ("emscad", "scrapped")]:
    evaluate_cross_source(train_source, test_source)


Distribusi label per sumber data:
label            0         1
source                      
emscad    0.578384  0.421616
scrapped  0.817243  0.182757

Seberapa mudah sumber data ditebak dari teks (source-prediction check):
              precision    recall  f1-score   support

      emscad       1.00      1.00      1.00       411
    scrapped       1.00      1.00      1.00       504

    accuracy                           1.00       915
   macro avg       1.00      1.00      1.00       915
weighted avg       1.00      1.00      1.00       915

Source-prediction accuracy: 100.00%

Fraud rate per source:
source
emscad      0.421616
scrapped    0.182757
Name: label, dtype: float64

Composition per label x source:
source  emscad  scrapped
label                   
0         1188      2057
1          866       460

Evaluasi Lintas Sumber: latih pada 'scrapped', uji pada 'emscad'
              precision    recall  f1-score   support

       valid       0.59      0.99      0.74      1188
     

## Kelas Trainer dan Metrik untuk IndoBERT

`ClassWeightedTrainer` menerapkan pembobotan kelas pada loss function. `compute_metrics` menggunakan `classification_report` untuk mencakup seluruh metrik (accuracy, precision, recall, f1-score per kelas, macro avg) dalam satu langkah, dengan `zero_division=0`. Fungsi `duplicate_minority` melakukan duplikasi data kelas minoritas sehingga setiap batch selama training memperoleh komposisi kelas yang relatif seimbang, melengkapi pembobotan loss.


In [ ]:
class ClassWeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    report = classification_report(labels, preds, labels=[0, 1], target_names=["valid", "fraud"], output_dict=True, zero_division=0)
    print(classification_report(labels, preds, labels=[0, 1], target_names=["valid", "fraud"], zero_division=0))
    return {
        "accuracy": float(report["accuracy"]),
        "precision_fraud": float(report["fraud"]["precision"]),
        "recall_fraud": float(report["fraud"]["recall"]),
        "macro_f1": float(report["macro avg"]["f1-score"])
    }

def duplicate_minority(df, label_col="label", seed=SEED):
    counts = df[label_col].value_counts()
    majority_count = counts.max()
    balanced_parts = []
    for label_value, count in counts.items():
        subset = df[df[label_col] == label_value]
        if count < majority_count:
            n_repeats = majority_count // count
            remainder = majority_count % count
            extra = subset.sample(remainder, random_state=seed) if remainder > 0 else subset.iloc[0:0]
            subset = pd.concat([subset] * n_repeats + [extra], ignore_index=True)
        balanced_parts.append(subset)
    return pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

## Fine-tuning IndoBERT dengan K-Fold Cross-Validation

Model utama adalah IndoBERT (`indobenchmark/indobert-base-p2`). Konfigurasi fine-tuning: max sequence length 256 token, learning rate 2e-5 dengan optimizer AdamW (weight decay 0,01, warmup 10%), batch size 16, maksimal 4 epoch dengan early-stop patience 1 berdasarkan macro-F1 pada data validasi, dan seed 42. K-Fold dijalankan pada data `scrapped` saja, sehingga setiap fold validasi murni berisi data `scrapped`; data `emscad` ditambahkan hanya pada bagian latih tiap fold. Setiap fold, data latih diseimbangkan melalui duplikasi kelas minoritas sebelum tokenisasi.

Teks yang masuk ke tahap ini (`text_clean_no_contact`) sudah melalui pembersihan tagar dan artefak narasi hoax di tahap preprocessing, sehingga jalan pintas sumber data yang paling kentara sudah dikurangi. Namun IndoBERT tetap berpotensi menangkap sinyal gaya penulisan yang lebih halus (mis. panjang teks, register formal vs informal) yang tidak bisa disaring lewat penghapusan kosakata seperti pada baseline klasik. Hasil evaluasi hold-out tetap perlu dibaca dengan hati-hati mengingat confound sumber data yang dijelaskan di atas.


In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text_clean_no_contact"], truncation=True, max_length=256, padding="max_length")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df_train_val_scrapped, df_train_val_scrapped["label"])):
    print(f"\n--- Training Fold {fold + 1} / 5 ---")

    # Val fold stays scrapped-only; emscad is added to the train fold only
    train_fold = pd.concat(
        [df_train_val_scrapped.iloc[train_idx], df_emscad], ignore_index=True
    )
    val_fold = df_train_val_scrapped.iloc[val_idx].reset_index(drop=True)

    train_fold_balanced = duplicate_minority(train_fold)

    class_counts = train_fold_balanced["label"].value_counts()
    total_samples = len(train_fold_balanced)
    weights = [total_samples / (2.0 * class_counts[0]), total_samples / (2.0 * class_counts[1])]
    class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

    ds_train = Dataset.from_pandas(train_fold_balanced).map(tokenize_function, batched=True)
    ds_val = Dataset.from_pandas(val_fold).map(tokenize_function, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=4,
        weight_decay=0.01,
        warmup_steps=0.10,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=SEED,
        logging_steps=10,
        report_to="none"
    )

    trainer = ClassWeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
    )

    trainer.train()
    eval_res = trainer.evaluate()
    fold_metrics.append(eval_res)

mean_macro_f1 = np.mean([m["eval_macro_f1"] for m in fold_metrics])
std_macro_f1 = np.std([m["eval_macro_f1"] for m in fold_metrics])
print(f"\n5-Fold CV Mean Macro F1: {mean_macro_f1:.4f} (+/- {std_macro_f1:.4f})")



--- Training Fold 1 / 5 ---


Map:   0%|          | 0/5008 [00:00<?, ? examples/s]

Map:   0%|          | 0/403 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

              precision    recall  f1-score   support

       valid       1.00      1.00      1.00       329
       fraud       0.99      0.99      0.99        73

    accuracy                           1.00       402
   macro avg       0.99      0.99      0.99       402
weighted avg       1.00      1.00      1.00       402



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       valid       1.00      1.00      1.00       329
       fraud       0.99      0.99      0.99        73

    accuracy                           1.00       402
   macro avg       0.99      0.99      0.99       402
weighted avg       1.00      1.00      1.00       402



Training Loss,Validation Loss,Epoch,Accuracy,Precision Fraud,Recall Fraud,Macro F1
0.001658,0.024040,3,0.995025,0.986301,0.986301,0.991631



5-Fold CV Mean Macro F1: 0.9917 (+/- 0.0052)


## Pelatihan Final dan Evaluasi Hold-out

Model final dilatih pada seluruh data `train+val` (setelah duplikasi kelas minoritas) dengan konfigurasi yang sama, kemudian dievaluasi pada hold-out test set. Hasil dibandingkan dengan target metrik: akurasi >= 0,85; presisi kelas penipuan >= 0,88; recall kelas penipuan >= 0,82; F1 makro >= 0,85.


In [ ]:
print("\n--- Final Training on Full Train+Val Dataset ---")
df_train_val_balanced = duplicate_minority(df_train_val)

class_counts_full = df_train_val_balanced["label"].value_counts()
total_full = len(df_train_val_balanced)
weights_full = [total_full / (2.0 * class_counts_full[0]), total_full / (2.0 * class_counts_full[1])]
class_weights_full = torch.tensor(weights_full, dtype=torch.float32)

ds_full_train = Dataset.from_pandas(df_train_val_balanced).map(tokenize_function, batched=True)
ds_test = Dataset.from_pandas(df_test).map(tokenize_function, batched=True)

final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

final_args = TrainingArguments(
    output_dir="./results_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=0.10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=10,
    report_to="none"
)

final_trainer = ClassWeightedTrainer(
    class_weights=class_weights_full,
    model=final_model,
    args=final_args,
    train_dataset=ds_full_train,
    eval_dataset=ds_test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

final_trainer.train()
test_results = final_trainer.evaluate(eval_dataset=ds_test)

targets_met = {
    "accuracy": test_results["eval_accuracy"] >= 0.85,
    "precision_fraud": test_results["eval_precision_fraud"] >= 0.88,
    "recall_fraud": test_results["eval_recall_fraud"] >= 0.82,
    "macro_f1": test_results["eval_macro_f1"] >= 0.85
}
print("\nHold-out Test Results:", json.dumps(test_results, indent=2))
print("Target Metrics Compliance:", json.dumps(targets_met, indent=2))


--- Final Training on Full Train+Val Dataset ---


Map:   0%|          | 0/5666 [00:00<?, ? examples/s]

Map:   0%|          | 0/504 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

              precision    recall  f1-score   support

       valid       1.00      1.00      1.00       412
       fraud       0.98      1.00      0.99        92

    accuracy                           1.00       504
   macro avg       0.99      1.00      0.99       504
weighted avg       1.00      1.00      1.00       504



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       valid       1.00      1.00      1.00       412
       fraud       0.98      1.00      0.99        92

    accuracy                           1.00       504
   macro avg       0.99      1.00      0.99       504
weighted avg       1.00      1.00      1.00       504



Training Loss,Validation Loss,Epoch,Accuracy,Precision Fraud,Recall Fraud,Macro F1
0.000195,0.036065,4,0.996032,0.978723,1.000000,0.993407



Hold-out Test Results: {
  "eval_loss": 0.036065101623535156,
  "eval_accuracy": 0.996031746031746,
  "eval_precision_fraud": 0.9787234042553191,
  "eval_recall_fraud": 1.0,
  "eval_macro_f1": 0.993407110901813
}
Target Metrics Compliance: {
  "accuracy": true,
  "precision_fraud": true,
  "recall_fraud": true,
  "macro_f1": true
}


## Menyimpan Artefak dan Versi Model

Setiap model yang lolos evaluasi disimpan pada direktori `models/indobert-v{N}/` berisi `model.safetensors`, `tokenizer/`, `metrics.json`, dan `card.md` yang mendeskripsikan konfigurasi training. Aktivasi model baru dilakukan secara manual melalui flag `model_artifacts.deployed=true` pada `deploy_config.json`, diikuti restart layanan API. Training dijalankan pada GPU sewaan, sedangkan inferensi produksi tetap berjalan di CPU VPS agar biaya operasional tetap rendah.


In [ ]:
VERSION = 1
export_dir = pathlib.Path(f"models/indobert-v{VERSION}")
tokenizer_dir = export_dir / "tokenizer"

export_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

final_trainer.model.save_pretrained(export_dir, safe_serialization=True)
tokenizer.save_pretrained(tokenizer_dir)

metrics_payload = {
    "model_version": f"v{VERSION}",
    "base_model": MODEL_NAME,
    "kfold_cv_mean_macro_f1": float(mean_macro_f1),
    "kfold_cv_std_macro_f1": float(std_macro_f1),
    "holdout_test_metrics": {
        "accuracy": float(test_results["eval_accuracy"]),
        "precision_fraud": float(test_results["eval_precision_fraud"]),
        "recall_fraud": float(test_results["eval_recall_fraud"]),
        "macro_f1": float(test_results["eval_macro_f1"])
    },
    "targets_met": targets_met,
    "baselines": baseline_metrics
}

with open(export_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

card_content = f"""# IndoBERT Job Fraud Classifier (v{VERSION})

## Model Description
Fine-tuned `indobenchmark/indobert-base-p2` for detecting fraudulent job postings in Indonesian language text.

## Training Configuration
- Max Sequence Length: 256
- Learning Rate: 2e-5 (AdamW, weight decay 0.01, warmup ratio 10%)
- Batch Size: 16
- Max Epochs: 4 (Early stopping patience = 1 on validation macro-F1)
- Seed: 42
- Class Balancing: Loss weight ratio + minority class duplication per batch

## Evaluation Metrics (Hold-out Test Set)
- Accuracy: {test_results['eval_accuracy']:.4f} (Target >= 0.85)
- Precision (Fraud): {test_results['eval_precision_fraud']:.4f} (Target >= 0.88)
- Recall (Fraud): {test_results['eval_recall_fraud']:.4f} (Target >= 0.82)
- Macro F1: {test_results['eval_macro_f1']:.4f} (Target >= 0.85)

## Deployment Status
model_artifacts.deployed=false (activate manually, then restart the api service)
"""

with open(export_dir / "card.md", "w", encoding="utf-8") as f:
    f.write(card_content)

deploy_config = {
    "model_artifacts": {
        "version": f"v{VERSION}",
        "path": str(export_dir),
        "deployed": False
    }
}

with open(export_dir / "deploy_config.json", "w", encoding="utf-8") as f:
    json.dump(deploy_config, f, indent=2)

print(f"Artifacts successfully saved to {export_dir.resolve()}")
print("Set model_artifacts.deployed=true in deploy_config.json and restart the api service to activate this model version.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Artifacts successfully saved to /content/models/indobert-v1
Set model_artifacts.deployed=true in deploy_config.json and restart the api service to activate this model version.


## Contoh Hasil Deteksi Akhir

Hasil akhir deteksi menggabungkan prediksi model, entitas yang diekstraksi (nomor telepon, email, nama perusahaan), dan `reasons[]` yang disusun dari fitur linguistik dan statistik.


In [ ]:
def build_detection_result(raw_text, model, tokenizer, device="cpu"):
    """Runs the full production pipeline on raw text: preprocess, predict, and explain."""
    processed = process_scraped_text(raw_text)

    model.eval()
    model.to(device)
    inputs = tokenizer(
        processed["text_clean_no_contact"], truncation=True, max_length=256,
        padding="max_length", return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_label = int(np.argmax(probs))

    def split_field(s):
      return [x.strip() for x in s.split(", ")] if s else []

    emails = split_field(processed["extracted_emails"])
    phones = split_field(processed["extracted_phones"])
    urls = split_field(processed["extracted_urls"])
    companies = split_field(processed["extracted_companies"])

    features = extract_entities_and_features(
        processed["text_clean_no_contact"],
        emails=emails, phones=phones, urls=urls, companies=companies
    )

    return {
        "predicted_label": "fraud" if pred_label == 1 else "valid",
        "confidence": float(probs[pred_label]),
        "extracted_phones": features["extracted_phones"],
        "extracted_emails": features["extracted_emails"],
        "extracted_urls": features["extracted_urls"],
        "extracted_companies": features["extracted_companies"],
        "reasons": features["reasons"]
    }

# Demo: run the production path on a raw, unprocessed sample from the test set
sample_row = df_test.iloc[3]
sample_result = build_detection_result(
    sample_row["text_utama"],
    final_trainer.model,
    tokenizer,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(json.dumps(sample_result, indent=2, ensure_ascii=False))

## Push Model ke Hugging Face Hub


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

HF_REPO_ID = "Kiuyha/indobert-job-fraud-detection"

print(f"Exporting model and tokenizer to Hugging Face Hub: {HF_REPO_ID}")

# Push model and tokenizer directly
final_trainer.model.push_to_hub(
    repo_id=HF_REPO_ID,
    commit_message=f"Upload fine-tuned IndoBERT job fraud model v{VERSION}",
    private=False
)
tokenizer.push_to_hub(
    repo_id=HF_REPO_ID,
    commit_message=f"Upload tokenizer for IndoBERT job fraud model v{VERSION}"
)

# Upload supplementary card.md and metrics.json to the repository root
api = HfApi()
api.upload_file(
    path_or_fileobj=str(export_dir / "metrics.json"),
    path_in_repo="metrics.json",
    repo_id=HF_REPO_ID,
    repo_type="model"
)
api.upload_file(
    path_or_fileobj=str(export_dir / "card.md"),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model"
)

print("Export to Hugging Face Hub complete.")

Exporting model and tokenizer to Hugging Face Hub: Kiuyha/indobert-job-fraud-detection


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...iu8jwuy/model.safetensors:   0%|          |  553kB /  498MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11689: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Export to Hugging Face Hub complete.
